# الدرس 02 - استكشاف إطار عمل عميل مايكروسوفت

إطار عمل **عميل مايكروسوفت (MAF)** هو إطار موحد لبناء وكلاء الذكاء الاصطناعي. يوفر بنية نظيفة وقابلة للتأليف بأربع كتل بناء أساسية:

- **العميل** – يتصل بنقطة نهاية نموذج الذكاء الاصطناعي ويتولى الاتصال
- **الوكيل** – يضم عميلًا مع التعليمات وتعريفات الأدوات
- **الأدوات** – توسع قدرات الوكيل باستخدام وظائف مخصصة يمكن للنموذج استدعاؤها
- **الجلسة** – يحتفظ بتاريخ المحادثة للتفاعلات متعددة الجولات

في هذا الدرس، سنبني **وكيل حجز سفر** يفحص توافر الوجهات باستخدام هذه المفاهيم.


## الإعداد


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## فهم بنية إطار العمل للوكيل

يتبع إطار عمل الوكيل من مايكروسوفت بنية متعددة الطبقات:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **العميل** – يربط `AzureAIProjectAgentProvider` بنشر Azure OpenAI. يتولى المصادقة، تنسيق الطلبات، وتحليل الردود.
2. **الوكيل** – يُنشأ من العميل عبر `provider.create_agent()`، يقوم الوكيل بدمج وصول النموذج مع التعليمات (موجه النظام) والأدوات.
3. **الأدوات** – دوال بايثون مزينة بـ `@tool` يمكن للوكيل استدعاؤها لأداء إجراءات أو استرجاع بيانات.
4. **الجلسة** – كائن `AgentSession` (يُنشأ عبر `agent.create_session()`) يخزن تاريخ المحادثة، مما يتيح حوارًا متعدد الأدوار حيث يتذكر الوكيل السياق السابق.

لنبنِ كل طبقة خطوة بخطوة.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## إضافة أدوات باستخدام المزخرف @tool

تتيح الأدوات للوكلاء اتخاذ إجراءات تتجاوز توليد النص. يقوم المزخرف `@tool` بتحويل دالة بايثون عادية إلى شيء يمكن للوكيل استدعاؤه.

النقاط الرئيسية:
- استخدم `Annotated[type, "الوصف"]` حتى يفهم النموذج كل معلمة.
- تصبح سلسلة التوثيق وصف الأداة الذي يراه النموذج.
- تعني `approval_mode="never_require"` أن الأداة تعمل تلقائيًا دون تأكيد المستخدم.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## إنشاء وكيل مع أدوات

الآن نقوم بدمج العميل والتعليمات والأدوات في وكيل. تعمل `instructions` كتعليمات النظام — فهي تحدد شخصية وسلوك الوكيل.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## محادثات متعددة الأدوار مع الجلسات

يحافظ `AgentSession` (الذي يتم إنشاؤه عبر `agent.create_session()`) على تتبع جميع الرسائل في المحادثة. من خلال تمرير نفس الجلسة إلى كل نداء `agent.run()`، يمكن للوكيل الوصول إلى سجل المحادثة الكامل والرجوع إلى الرسائل السابقة.

نمرر `tools=[check_destination_availability]` بحيث يمكن للوكيل استدعاء أداة التحقق من التوفر الخاصة بنا خلال كل دور.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## الملخص

في هذا الدرس استكشفت الأعمدة الأربعة لإطار عمل وكيل مايكروسوفت:

| المفهوم | ما تعلمته |
|---------|------------------|
| **العميل** | `AzureAIProjectAgentProvider` يتصل بـ Azure OpenAI باستخدام المصادقة المعتمدة على بيانات الاعتماد |
| **الوكيل** | `provider.create_agent()` يجمع اتصال نموذج مع التعليمات واسم |
| **الأدوات** | المُزخرف `@tool` يتيح دوال بيثون ليتم استدعاؤها من قبل الوكيل |
| **الجلسة** | `agent.create_session()` يحافظ على تاريخ المحادثة عبر جولات متعددة |

تتجمع هذه اللبنات الأساسية معًا لإنشاء وكلاء يمكنهم إجراء محادثات طبيعية، استدعاء دوال خارجية، والحفاظ على السياق — وهو الأساس لأنماط وكيل أكثر تقدمًا في الدروس اللاحقة.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:  
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المعتمد والموثوق. للمعلومات الهامة والحرجة، يُنصح بالاعتماد على الترجمة المهنية البشرية. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
